In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

## Chat completions API

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

In [6]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.model_dump_json()

'{"id":"chatcmpl-E8yxG77A35XNyMJ2NLcyX0UO4o6ub","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"Hi there! Nice to meet you, and welcome—thanks for saying hi!\\n\\nI’m here to help with lots of things. A few ideas of what I can do:\\n- Explain topics in simple terms\\n- Help you write or edit emails, messages, essays, etc.\\n- Brainstorm ideas or plan a project or trip\\n- Learn a new skill with step-by-step guidance\\n- Code help or debugging\\n- Translate or practice a language\\n- Have a fun chat, tell stories, or play quick games\\n\\nWhat would you like to start with today? You can ask me a question, or tell me a goal you have and I’ll help you get there.","refusal":null,"role":"assistant","annotations":[],"audio":null,"function_call":null,"tool_calls":null}}],"created":1785809642,"model":"gpt-5-nano-2025-08-07","object":"chat.completion","service_tier":"default","system_fingerprint":null,"usage":{"completion_tokens":849,"prompt_tokens":21,"total_

In [7]:
response.choices[0].message.content

'Hi there! Nice to meet you, and welcome—thanks for saying hi!\n\nI’m here to help with lots of things. A few ideas of what I can do:\n- Explain topics in simple terms\n- Help you write or edit emails, messages, essays, etc.\n- Brainstorm ideas or plan a project or trip\n- Learn a new skill with step-by-step guidance\n- Code help or debugging\n- Translate or practice a language\n- Have a fun chat, tell stories, or play quick games\n\nWhat would you like to start with today? You can ask me a question, or tell me a goal you have and I’ll help you get there.'

## Another way of calling LLM via direct endpoint

In [8]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a joke about zionism."}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a joke about zionism.'}]}

In [9]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-E8yyPzpl8BzVQUOBnFmrk8IWIT17j',
 'object': 'chat.completion',
 'created': 1785809713,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'I’d like to keep humor respectful when it touches political topics. Here are a couple light, neutral jokes about geography/p borders instead:\n\n- Why did the atlas apply for a job? It wanted to help everyone find their place.\n- Why do borders make great comedians? They always draw the line.\n\nWould you like a different style or topic for a joke?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 15,
  'completion_tokens': 2642,
  'total_tokens': 2657,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 2560,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint'

In [10]:
response.json()["choices"][0]["message"]["content"]

'I’d like to keep humor respectful when it touches political topics. Here are a couple light, neutral jokes about geography/p borders instead:\n\n- Why did the atlas apply for a job? It wanted to help everyone find their place.\n- Why do borders make great comedians? They always draw the line.\n\nWould you like a different style or topic for a joke?'

## Types of roles in messages

- **A system prompt** that tells them what task they are performing and what tone they should use
- **A user prompt** -- the conversation starter that they should reply to

In [11]:
messages = [
    {"role": "system", "content": "You are a Ricky Gervais."},
    {"role": "user", "content": "Tell me a new original joke."}
    ]

openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

"Here’s a new joke in the spirit of Ricky Gervais (without claiming to be him):\n\nLife's a rollercoaster, but my Wi‑Fi is the real ride: it pretends to be fast, then buffers at the worst moment and makes me question all my life choices."

### The Illusion of "memory"

In [34]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Amit!"}
    ]
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Hello Amit! How can I assist you today?'

### OK let's now ask a follow-up question

In [35]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'I’m sorry, but I don’t know your name. Could you please tell me?'

## Maintaining history

In [15]:
from copy import deepcopy

messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

# User asks first question
messages.append({"role": "user", "content": "What is 10 + 20?"})

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)

answer = response.choices[0].message

answer

ChatCompletionMessage(content='10 + 20 = 30', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

### Save a checkpoint

In [16]:
messages.append({
    "role": "assistant",
    "content": answer.content
})
checkpoint = deepcopy(messages)
checkpoint

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'}]

### Continue Branch A

In [ ]:
messages.append({
    "role": "user",
    "content": "Multiply that by 5."
})

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)

messages.append(response.choices[0].message)
messages

In [18]:
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'},
 {'role': 'user', 'content': 'Multiply that by 5.'},
 ChatCompletionMessage(content='30 multiplied by 5 is 150.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)]

### Continue further

In [20]:
messages.append({
    "role": "user",
    "content": "Subtract 20."
})

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)


messages.append(response.choices[0].message)
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'},
 {'role': 'user', 'content': 'Multiply that by 5.'},
 ChatCompletionMessage(content='30 multiplied by 5 is 150.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None),
 {'role': 'user', 'content': 'Subtract 20.'},
 ChatCompletionMessage(content='150 minus 20 is 130.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None),
 {'role': 'user', 'content': 'Subtract 20.'},
 ChatCompletionMessage(content='130 minus 20 is 110.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)]

### Go back to the old conversation

In [21]:
messages = deepcopy(checkpoint)
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'}]

In [23]:
messages.append({
    "role": "user",
    "content": "Multiply that by 8."
})

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)

messages.append(response.choices[0].message)
response.choices[0].message

ChatCompletionMessage(content='240 multiplied by 8 is 1920.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)